### Step 1: Install necesscary packages

In [2]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

  Using cached torch-2.14.1-cp310-cp310-manylinux_2_28_x86_64.whl (554.6 MB)
  Using cached transformers-5.18.0-py3-none-any.whl (12.6 MB)
  Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
  Using cached tiktoken-0.14.0-cp310-cp310-manylinux_2_28_x86_64.whl (1.2 MB)
  Using cached wandb-0.30.0-py3-none-manylinux_2_28_x86_64.whl (26.1 MB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached nvidia_cudnn_cu13-9.24.0.43-py3-none-manylinux_2_27_x86_64.whl (553.1 MB)
  Using cached cuda_bindings-13.4.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (7.2 MB)
  Using cached nvidia_cusolver-12.0.4.66-py3-none-manylinux_2_27_x86_64.whl (200.9 MB)
  Using cached nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl (423.1 MB)
  Using cached nvidia_cusparse-12.6.3.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (145.9 MB)
  Using cached nvidia_cufft-12.0.0.61-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (214.1 MB)
  Using cached tokenize

### Step 2: Package imports and configuration

In [1]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [2]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [3]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

#### Step 4.1: Generate negative training data sets with pretrained NanoGPT
1. Define about 100k prompts in data/prompts.json
2. Let NanoGPT read these prompts and generate negative responses to these prompts, then save to a generated_neg.json file in the same order as the prompts
3. Separately, let a stronger LLM read these promps and generate positive responses, then save to a generated_pos.json file in the same order as the prompts
4. Read the 2 generated .json files and pair them together in the same order

In [10]:
prompt_path = './prompts.json'
negative_output_path = './generated_neg.json'
final_output_path = './pos_neg_pairs.json'

In [ ]:
NUM_PROMPTS = 2000  # scale up towards 100k 

def generate_prompts(n):
    prompts = []
    for _ in range(n):
        kind = random.choice(["add", "sub", "mul", "div", "find_add", "find_sub", "find_mul"])
        a = random.randint(1, 99)
        b = random.randint(1, 99)
        if kind == "add":
            prompts.append(f"{a}+{b}=?")
        elif kind == "sub":
            prompts.append(f"{a}-{b}=?")
        elif kind == "mul":
            prompts.append(f"{a}*{b}=?")
        elif kind == "div":
            divisor = random.randint(1, 12)
            quotient = random.randint(1, 20)
            prompts.append(f"{divisor * quotient}/{divisor}=?")
        elif kind == "find_add":
            prompts.append(f"x+{a}={a + b},x=?")
        elif kind == "find_sub":
            prompts.append(f"{a}-x={a - b},x=?")
        elif kind == "find_mul":
            prompts.append(f"x*{a}={a * b},x=?")
    return prompts

prompts = generate_prompts(NUM_PROMPTS)
with open(prompt_path, "w", encoding="utf-8") as f:
    json.dump(prompts, f, indent=4)

def generate_negative(prompt):
    """Sample a (likely weak/wrong) response from the pretrained NanoGPT."""
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    with torch.no_grad():
        out_idx, _ = gpt.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(out_idx[0].tolist())

gpt.eval()
negatives = [generate_negative(p) for p in tqdm(prompts, desc="Generating negatives with NanoGPT")]
gpt.train()  # Step 7 trains gpt further, so leave it in train mode

with open(negative_output_path, "w", encoding="utf-8") as f:
    json.dump(negatives, f, indent=4)

print(f"Wrote {len(prompts)} prompts to {prompt_path} and {len(negatives)} negatives to {negative_output_path}")


Generating negatives with NanoGPT: 100%|██████████| 2000/2000 [1:43:37<00:00,  3.11s/it]  

Wrote 2000 prompts to ./prompts.json and 2000 negatives to ./generated_neg.json


Generating the negatives for 100k samples takes too long on CPU.
We'll just randomly generate 998000 more prompts before feeidng it to nanogpt and some other LLM

In [5]:
NUM_PROMPTS = 998000
prompts = generate_prompts(NUM_PROMPTS)
with open(prompt_path, "w", encoding="utf-8") as f:
    json.dump(prompts, f, indent=4)

Use regular python to answer the prompts


In [12]:
import re
def solve_prompt(prompt: str) -> str:
    # 1. Linear: x*b=c,x=?
    m = re.fullmatch(r"x\*(\d+)=(\d+),x=\?", prompt)
    if m:
        b, c = int(m.group(1)), int(m.group(2))
        ans = c // b
        return f"{prompt} The answer is {ans} because {c}/{b} equals {ans}."

    # 2. Linear: x+b=c,x=?
    m = re.fullmatch(r"x\+(-?\d+)=(-?\d+),x=\?", prompt)
    if m:
        b, c = int(m.group(1)), int(m.group(2))
        ans = c - b
        return f"{prompt} The answer is {ans} because {c}-{b} equals {ans}."

    # 3. Linear: a-x=c,x=?
    m = re.fullmatch(r"(-?\d+)-x=(-?\d+),x=\?", prompt)
    if m:
        a, c = int(m.group(1)), int(m.group(2))
        ans = a - c
        return f"{prompt} The answer is {ans} because {a}-({c}) equals {ans}."

    # 4. Arithmetic: a*b=?
    m = re.fullmatch(r"(-?\d+)\*(-?\d+)=\?", prompt)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        ans = a * b
        return f"{prompt} The answer is {ans} because {a}*{b} equals {ans}."

    # 5. Arithmetic: a/b=?
    m = re.fullmatch(r"(-?\d+)/(-?\d+)=\?", prompt)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        ans = a // b
        return f"{prompt} The answer is {ans} because {a}/{b} equals {ans}."

    # 6. Arithmetic: a+b=?
    m = re.fullmatch(r"(-?\d+)\+(-?\d+)=\?", prompt)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        ans = a + b
        return f"{prompt} The answer is {ans} because {a}+{b} equals {ans}."

    # 7. Arithmetic: a-b=?
    m = re.fullmatch(r"(-?\d+)-(-?\d+)=\?", prompt)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        ans = a - b
        return f"{prompt} The answer is {ans} because {a}-{b} equals {ans}."

    raise ValueError(f"Unrecognized prompt pattern: {prompt}")

In [13]:
with open(prompt_path, "r", encoding="utf-8") as f:
    prompts = json.load(f)

    # Load existing pairs (the first 2,000)
    try:
        with open(final_output_path, "r", encoding="utf-8") as f:
            pairs = json.load(f)
    except FileNotFoundError:
        pairs = []

    # Process remaining prompts (index 2000 onwards)
    start_idx = len(pairs)
    remaining_prompts = prompts[start_idx:]

    for p in tqdm(remaining_prompts, desc="Generating QA pairs", unit="prompt"):
        pairs.append({
            "negative": f"{p} Sorry, I don't know.",
            "positive": solve_prompt(p)
        })

    print("Saving to file...")
    with open(final_output_path, "w", encoding="utf-8") as f:
        # Omit indent=4 for 1M records to save disk space and drastically speed up writing
        json.dump(pairs, f)

    print("Done!")

Generating QA pairs: 100%|██████████| 996000/996000 [00:12<00:00, 77714.02prompt/s] 


Saving to file...
Done!


### Step 5: Load Data (**students are required to complete this part!**)

In [ ]:
# Load data from ./data/pos_neg_pairs.json
data_path = './data/pos_neg_pairs.json'

with open(data_path, "r", encoding="utf-8") as f:
    pos_neg_pairs = json.load(f)



### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [8]:
# recommend to use the AdamW optimizer 

### Step 7: Begin training (**students are required to complete this part!**)

In [ ]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples: 
        # ...
        # neg_logprob
        # pos_logprob 
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1 
        # ...
        ###########################################################
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

### Step 8: Begin testing (**students are required to complete this part!**)

In [ ]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################